# Notebook 2 — Preprocessing Pipeline

Goal: build and validate the full preprocessing pipeline, save processed arrays to `data/processed/`.

In [1]:
import numpy as np
import os
import sys
sys.path.append('..')
from src.preprocessing import (
    load_raw, add_rul, normalize_standard, normalize_clustered,
    make_windows, make_test_windows, get_feature_cols
)

DATA_DIR = '../data/raw'
OUT_DIR = '../data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

WINDOW_SIZE = 30
MULTI_COND = {'FD002', 'FD004'}

In [2]:
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    train_df = load_raw(f'{DATA_DIR}/train_{fd}.txt')
    test_df  = load_raw(f'{DATA_DIR}/test_{fd}.txt')
    rul_test = np.loadtxt(f'{DATA_DIR}/RUL_{fd}.txt')

    train_df = add_rul(train_df)
    feat_cols = get_feature_cols(train_df)

    if fd in MULTI_COND:
        train_df, test_df, _ = normalize_clustered(train_df, test_df, feat_cols)
    else:
        train_df, test_df, _ = normalize_standard(train_df, test_df, feat_cols)

    X_train, y_train = make_windows(train_df, feat_cols, WINDOW_SIZE)
    X_test = make_test_windows(test_df, feat_cols, WINDOW_SIZE)

    np.save(f'{OUT_DIR}/X_train_{fd}.npy', X_train)
    np.save(f'{OUT_DIR}/y_train_{fd}.npy', y_train)
    np.save(f'{OUT_DIR}/X_test_{fd}.npy',  X_test)
    np.save(f'{OUT_DIR}/y_test_{fd}.npy',  rul_test.astype(np.float32))

    print(f'{fd}: X_train={X_train.shape}, y_train={y_train.shape}, X_test={X_test.shape}')

FD001: X_train=(17731, 30, 17), y_train=(17731,), X_test=(100, 30, 17)
FD002: X_train=(46219, 30, 17), y_train=(46219,), X_test=(259, 30, 17)
FD003: X_train=(21820, 30, 17), y_train=(21820,), X_test=(100, 30, 17)
FD004: X_train=(54028, 30, 17), y_train=(54028,), X_test=(248, 30, 17)
